# Deteksi Inkonsistensi Sentimen dan Rating pada Review Aplikasi Mobile Indonesia

Notebook ini mencakup dua bagian utama:
1. **Data Collection** - Scraping review dari Google Play Store
2. **Preprocessing** - Pembersihan dan transformasi teks

## Bagian 1: Data Collection

### 1.1 Instalasi Library

In [17]:
import subprocess
import sys

packages = [
    'google-play-scraper',
    'pandas',
    'numpy',
    'langdetect',
    'tqdm'
]

for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Semua library berhasil diinstall.')

Installing google-play-scraper...
Installing pandas...
Installing numpy...
Installing langdetect...
Installing tqdm...
Semua library berhasil diinstall.


### 1.2 Import Library

In [18]:
import pandas as pd
import numpy as np
import re
import time
import warnings
from datetime import datetime

from google_play_scraper import reviews, Sort
from langdetect import detect, LangDetectException
from tqdm import tqdm

warnings.filterwarnings('ignore')

print('Semua library berhasil diimport.')
print(f'Pandas version  : {pd.__version__}')
print(f'Numpy version   : {np.__version__}')

Semua library berhasil diimport.
Pandas version  : 3.0.1
Numpy version   : 2.4.6


### 1.3 Konfigurasi Aplikasi Target

In [19]:
APPS = [
    {'name': 'Gojek',      'app_id': 'com.gojek.app'},
    {'name': 'Tokopedia',  'app_id': 'com.tokopedia.tkpd'},
    {'name': 'Shopee',     'app_id': 'com.shopee.id'},
    {'name': 'DANA',       'app_id': 'id.dana'},
    {'name': 'BCA Mobile', 'app_id': 'com.bca'},
]

SCRAPE_COUNT   = 5000
MAX_RETRY      = 3
DELAY_SECONDS  = 2
MIN_TOTAL      = 10000
EXTRA_COUNT    = 500

print(f'Total aplikasi target : {len(APPS)}')
print(f'Review per aplikasi   : {SCRAPE_COUNT:,}')
print(f'Target total review   : {SCRAPE_COUNT * len(APPS):,} (sebelum filtering)')

Total aplikasi target : 5
Review per aplikasi   : 5,000
Target total review   : 25,000 (sebelum filtering)


### 1.4 Fungsi Scraping dengan Retry Logic

In [20]:
def scrape_app_reviews(app_name, app_id, count=2500, max_retry=3):
    """
    Scrape review dari satu aplikasi dengan retry logic.
    Mengembalikan list of dict atau list kosong jika gagal.
    """
    for attempt in range(1, max_retry + 1):
        try:
            print(f'  Scraping {app_name} ({app_id}) - percobaan {attempt}/{max_retry}...')
            result, _ = reviews(
                app_id,
                lang='id',
                country='id',
                sort=Sort.NEWEST,
                count=count
            )

            records = []
            for r in result:
                records.append({
                    'app':          app_name,
                    'app_id':       app_id,
                    'username':     r.get('userName', ''),
                    'rating':       r.get('score', np.nan),
                    'text':         r.get('content', ''),
                    'thumbs_up':    r.get('thumbsUpCount', 0),
                    'date':         r.get('at', None),
                    'reply':        r.get('replyContent', ''),
                })

            print(f'  Berhasil mengambil {len(records):,} review dari {app_name}.')
            return records

        except Exception as e:
            print(f'  Gagal (percobaan {attempt}): {e}')
            if attempt < max_retry:
                print(f'  Menunggu {DELAY_SECONDS} detik sebelum retry...')
                time.sleep(DELAY_SECONDS)

    print(f'  PERINGATAN: Gagal scraping {app_name} setelah {max_retry} percobaan.')
    return []


print('Fungsi scraping siap.')

Fungsi scraping siap.


### 1.5 Proses Scraping Utama

In [21]:
all_records = []
app_record_counts = {}

print('=' * 60)
print('MEMULAI PROSES SCRAPING')
print('=' * 60)

for app in tqdm(APPS, desc='Progress scraping'):
    print(f'\nMulai scraping: {app["name"]}')
    records = scrape_app_reviews(
        app_name=app['name'],
        app_id=app['app_id'],
        count=SCRAPE_COUNT,
        max_retry=MAX_RETRY
    )
    all_records.extend(records)
    app_record_counts[app['name']] = len(records)
    print(f'Total review terkumpul sejauh ini: {len(all_records):,}')
    time.sleep(DELAY_SECONDS)

print('\n' + '=' * 60)
print(f'SCRAPING SELESAI. Total raw review: {len(all_records):,}')
print('=' * 60)

for app_name, count in app_record_counts.items():
    print(f'  {app_name:<15}: {count:,} review')

MEMULAI PROSES SCRAPING


Progress scraping:   0%|          | 0/5 [00:00<?, ?it/s]


Mulai scraping: Gojek
  Scraping Gojek (com.gojek.app) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari Gojek.
Total review terkumpul sejauh ini: 5,000


Progress scraping:  20%|██        | 1/5 [00:04<00:16,  4.23s/it]


Mulai scraping: Tokopedia
  Scraping Tokopedia (com.tokopedia.tkpd) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari Tokopedia.
Total review terkumpul sejauh ini: 10,000


...


Mulai scraping: DANA
  Scraping DANA (id.dana) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari DANA.
Total review terkumpul sejauh ini: 20,000


Progress scraping:  80%|████████  | 4/5 [00:19<00:04,  4.77s/it]


Mulai scraping: BCA Mobile
  Scraping BCA Mobile (com.bca) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari BCA Mobile.
Total review terkumpul sejauh ini: 25,000


Progress scraping: 100%|██████████| 5/5 [00:24<00:00,  4.81s/it]


SCRAPING SELESAI. Total raw review: 25,000
  Gojek          : 5,000 review
  Tokopedia      : 5,000 review
  Shopee         : 5,000 review
  DANA           : 5,000 review
  BCA Mobile     : 5,000 review


### 1.6 Simpan Raw Data

In [22]:
df_raw = pd.DataFrame(all_records)

print(f'Shape raw DataFrame  : {df_raw.shape}')
print(f'Kolom yang tersedia  : {list(df_raw.columns)}')
print('\nContoh data (5 baris pertama):')
display(df_raw.head())

df_raw.to_csv('raw_reviews.csv', index=False, encoding='utf-8-sig')
print('\nRaw data berhasil disimpan ke raw_reviews.csv')

Shape raw DataFrame  : (25000, 8)
Kolom yang tersedia  : ['app', 'app_id', 'username', 'rating', 'text', 'thumbs_up', 'date', 'reply']

Contoh data (5 baris pertama):


,app,app_id,username,rating,text,thumbs_up,date,reply
0,Gojek,com.gojek.app,Pengguna Google,5,sangat membantu,0,2026-05-30 15:49:54,NaN
1,Gojek,com.gojek.app,Pengguna Google,5,aplikasi bagus bangga buatan Indonesia,0,2026-05-30 15:36:01,NaN
2,Gojek,com.gojek.app,Pengguna Google,5,saya pernah membayar 2 kali dan dikembalikan🥳 ...,0,2026-05-30 15:22:14,NaN
3,Gojek,com.gojek.app,Pengguna Google,5,sangat puas naik gojek,0,2026-05-30 15:18:35,NaN
4,Gojek,com.gojek.app,Pengguna Google,5,"gojek emang nyaman kemana "" aman",0,2026-05-30 15:16:47,NaN



Raw data berhasil disimpan ke raw_reviews.csv


## Bagian 2: Preprocessing

### 2.1 Load Data dan Inisialisasi

In [23]:
df = df_raw.copy()

print(f'Data dimuat: {len(df):,} baris, {df.shape[1]} kolom')
print('\nInfo kolom:')
print(df.dtypes)
print('\nJumlah missing values per kolom:')
print(df.isnull().sum())

Data dimuat: 25,000 baris, 8 kolom

Info kolom:
app                     str
app_id                  str
username                str
rating                int64
text                    str
thumbs_up             int64
date         datetime64[us]
reply                   str
dtype: object

Jumlah missing values per kolom:
app              0
app_id           0
username         0
rating           0
text             0
thumbs_up        0
date             0
reply        12609
dtype: int64


### 2.2 Filter Review Kosong dan Terlalu Pendek

In [24]:
before = len(df)

# Hapus review dengan teks NaN atau kosong
df = df[df['text'].notna()]
df = df[df['text'].str.strip() != '']

# Hapus review dengan kurang dari 5 kata
df['_word_count_temp'] = df['text'].apply(lambda x: len(str(x).split()))
df = df[df['_word_count_temp'] >= 5]
df = df.drop(columns=['_word_count_temp'])
df = df.reset_index(drop=True)

after = len(df)
print(f'Filter teks kosong/pendek:')
print(f'  Sebelum : {before:,}')
print(f'  Sesudah : {after:,}')
print(f'  Dihapus : {before - after:,} review')

Filter teks kosong/pendek:
  Sebelum : 25,000
  Sesudah : 13,653
  Dihapus : 11,347 review


### 2.3 Filter Bahasa Indonesia

In [25]:
before = len(df)

def detect_language(text):
    try:
        lang = detect(str(text))
        return lang
    except LangDetectException:
        return 'unknown'
    except Exception:
        return 'unknown'

tqdm.pandas(desc='Deteksi bahasa')
df['lang'] = df['text'].progress_apply(detect_language)

lang_dist = df['lang'].value_counts()
print('\nDistribusi bahasa yang terdeteksi:')
print(lang_dist.head(10))

df = df[df['lang'] == 'id'].reset_index(drop=True)

after = len(df)
print(f'\nFilter bahasa Indonesia:')
print(f'  Sebelum : {before:,}')
print(f'  Sesudah : {after:,}')
print(f'  Dihapus : {before - after:,} review (bukan bahasa Indonesia)')

Deteksi bahasa: 100%|██████████| 13653/13653 [00:19<00:00, 683.09it/s]


Distribusi bahasa yang terdeteksi:
lang
id    12801
tl      343
de      191
en       99
et       31
so       25
hr       19
no       16
fi       13
ca       10
Name: count, dtype: int64

Filter bahasa Indonesia:
  Sebelum : 13,653
  Sesudah : 12,801
  Dihapus : 852 review (bukan bahasa Indonesia)


### 2.4 Normalisasi Teks

In [26]:
def normalize_text(text):
    text = str(text)
    # Lowercase
    text = text.lower()
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Hapus mention (@username)
    text = re.sub(r'@\w+', '', text)
    # Hapus karakter yang berulang lebih dari 2 kali berturut-turut
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Hapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Simpan teks asli sebelum normalisasi
df['text_original'] = df['text']

tqdm.pandas(desc='Normalisasi teks')
df['text'] = df['text'].progress_apply(normalize_text)

print('Normalisasi teks selesai.')
print('\nContoh sebelum dan sesudah normalisasi (5 sampel):')
sample_idx = df.sample(5, random_state=42).index
for i in sample_idx:
    print(f'  Asli  : {df.loc[i, "text_original"][:80]}')
    print(f'  Norma : {df.loc[i, "text"][:80]}')
    print()

Normalisasi teks: 100%|██████████| 12801/12801 [00:00<00:00, 165272.10it/s]

Normalisasi teks selesai.

Contoh sebelum dan sesudah normalisasi (5 sampel):
  Asli  : bagus banget lengkap shopee cuman saran ya bikin lebih lengkap dong permainan ny
  Norma : bagus banget lengkap shopee cuman saran ya bikin lebih lengkap dong permainan ny

  Asli  : aplikasi babi. duit di saldo gk bisa di buka. ilang mental terus.
  Norma : aplikasi babi. duit di saldo gk bisa di buka. ilang mental terus.

  Asli  : Gojek menyelesaikan masalah tanpa masalah
  Norma : gojek menyelesaikan masalah tanpa masalah

  Asli  : sangat puas dengan aplikasih ini.
  Norma : sangat puas dengan aplikasih ini.

  Asli  : Ini adminnya beneran manusia atau bot sih? Kok responnya gak ada sama sekali
  Norma : ini adminnya beneran manusia atau bot sih? kok responnya gak ada sama sekali



### 2.5 Hapus Duplikat

In [27]:
before = len(df)

df = df.drop_duplicates(subset=['text', 'app'], keep='first').reset_index(drop=True)

after = len(df)
print(f'Hapus duplikat (berdasarkan text + app):')
print(f'  Sebelum : {before:,}')
print(f'  Sesudah : {after:,}')
print(f'  Dihapus : {before - after:,} duplikat')

Hapus duplikat (berdasarkan text + app):
  Sebelum : 12,801
  Sesudah : 12,792
  Dihapus : 9 duplikat


### 2.6 Tambah Kolom word_count

In [28]:
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

print('Kolom word_count berhasil ditambahkan.')
print(f'\nStatistik word_count:')
print(df['word_count'].describe().round(2))

Kolom word_count berhasil ditambahkan.

Statistik word_count:
count    12792.00
mean        19.09
std         15.95
min          5.00
25%          8.00
50%         14.00
75%         24.00
max         98.00
Name: word_count, dtype: float64


### 2.7 Tambah Kolom star_sentiment

In [29]:
def map_star_sentiment(rating):
    if rating in [4, 5]:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    elif rating in [1, 2]:
        return 'negative'
    else:
        return 'unknown'

df['star_sentiment'] = df['rating'].apply(map_star_sentiment)

print('Kolom star_sentiment berhasil ditambahkan.')
print('\nDistribusi star_sentiment:')
print(df['star_sentiment'].value_counts())
print(f'\nProporsi:')
print((df['star_sentiment'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

Kolom star_sentiment berhasil ditambahkan.

Distribusi star_sentiment:
star_sentiment
negative    6898
positive    4931
neutral      963
Name: count, dtype: int64

Proporsi:
star_sentiment
negative    53.92%
positive    38.55%
neutral      7.53%
Name: proportion, dtype: str


### 2.8 Tambah Kolom rough_text_sentiment (Keyword Matching)

In [30]:
POSITIVE_KEYWORDS = [
    'bagus', 'baik', 'mantap', 'keren', 'hebat', 'luar biasa', 'memuaskan',
    'mudah', 'cepat', 'lancar', 'bermanfaat', 'membantu', 'praktis', 'nyaman',
    'recommended', 'rekomen', 'puas', 'senang', 'suka', 'top', 'oke', 'ok',
    'terbaik', 'sempurna', 'canggih', 'andalan', 'favorit', 'sip', 'mulus'
]

NEGATIVE_KEYWORDS = [
    'buruk', 'jelek', 'payah', 'parah', 'kecewa', 'mengecewakan', 'lambat',
    'lemot', 'error', 'crash', 'bug', 'masalah', 'susah', 'sulit', 'gagal',
    'tidak bisa', 'gabisa', 'tidak berguna', 'percuma', 'rugi', 'boros',
    'menipu', 'tipu', 'penipuan', 'bohong', 'sampah', 'mempersulit', 'ribet',
    'lama', 'macet', 'down', 'tidak jelas', 'membingungkan', 'bermasalah'
]

print(f'Kata kunci positif  : {len(POSITIVE_KEYWORDS)} kata')
print(f'Kata kunci negatif  : {len(NEGATIVE_KEYWORDS)} kata')

def keyword_sentiment(text):
    text_lower = str(text).lower()
    pos_score = sum(1 for kw in POSITIVE_KEYWORDS if kw in text_lower)
    neg_score = sum(1 for kw in NEGATIVE_KEYWORDS if kw in text_lower)

    if pos_score > neg_score:
        return 'positive'
    elif neg_score > pos_score:
        return 'negative'
    else:
        return 'neutral'

tqdm.pandas(desc='Keyword sentiment')
df['rough_text_sentiment'] = df['text'].progress_apply(keyword_sentiment)

print('\nDistribusi rough_text_sentiment:')
print(df['rough_text_sentiment'].value_counts())

Kata kunci positif  : 29 kata
Kata kunci negatif  : 34 kata


Keyword sentiment: 100%|██████████| 12792/12792 [00:00<00:00, 159035.64it/s]


Distribusi rough_text_sentiment:
rough_text_sentiment
neutral     4780
positive    4295
negative    3717
Name: count, dtype: int64


### 2.9 Tambah Kolom is_mismatch

In [31]:
def compute_mismatch(row):
    star  = row['star_sentiment']
    text  = row['rough_text_sentiment']
    # Mismatch: keduanya bukan neutral, dan label berbeda
    if star == 'neutral' or text == 'neutral':
        return 0
    return 1 if star != text else 0

df['is_mismatch'] = df.apply(compute_mismatch, axis=1)

n_mismatch = df['is_mismatch'].sum()
pct_mismatch = n_mismatch / len(df) * 100

print('Kolom is_mismatch berhasil ditambahkan.')
print(f'\nTotal review mismatch  : {n_mismatch:,} ({pct_mismatch:.2f}%)')
print(f'Total review konsisten : {len(df) - n_mismatch:,} ({100 - pct_mismatch:.2f}%)')

Kolom is_mismatch berhasil ditambahkan.

Total review mismatch  : 1,753 (13.70%)
Total review konsisten : 11,039 (86.30%)


### 2.10 Validasi Total Review dan Scraping Tambahan

In [32]:
iteration = 0

while len(df) < MIN_TOTAL:
    iteration += 1
    print('=' * 60)
    print(f'PERINGATAN: Total review setelah preprocessing = {len(df):,}')
    print(f'Kurang dari minimum {MIN_TOTAL:,}. Memulai iterasi tambahan ke-{iteration}...')
    print('=' * 60)

    # Identifikasi 2 aplikasi dengan review paling sedikit
    counts_per_app = df['app'].value_counts()
    apps_sorted    = counts_per_app.sort_values().head(2).index.tolist()

    print(f'\nAplikasi yang akan di-scrape tambahan: {apps_sorted}')

    new_records = []
    for app_name in apps_sorted:
        app_cfg = next((a for a in APPS if a['name'] == app_name), None)
        if app_cfg is None:
            continue
        print(f'\nScraping tambahan {EXTRA_COUNT} review untuk {app_name}...')
        extra = scrape_app_reviews(
            app_name=app_cfg['name'],
            app_id=app_cfg['app_id'],
            count=EXTRA_COUNT,
            max_retry=MAX_RETRY
        )
        new_records.extend(extra)
        time.sleep(DELAY_SECONDS)

    if not new_records:
        print('Tidak ada review tambahan yang berhasil diambil. Menghentikan loop.')
        break

    # Preprocessing review tambahan
    df_extra = pd.DataFrame(new_records)
    df_extra = df_extra[df_extra['text'].notna()]
    df_extra = df_extra[df_extra['text'].str.strip() != '']
    df_extra['_wc'] = df_extra['text'].apply(lambda x: len(str(x).split()))
    df_extra = df_extra[df_extra['_wc'] >= 5].drop(columns=['_wc'])

    tqdm.pandas(desc='Deteksi bahasa (tambahan)')
    df_extra['lang'] = df_extra['text'].progress_apply(detect_language)
    df_extra = df_extra[df_extra['lang'] == 'id']

    df_extra['text_original'] = df_extra['text']
    tqdm.pandas(desc='Normalisasi teks (tambahan)')
    df_extra['text'] = df_extra['text'].progress_apply(normalize_text)

    df_extra['word_count']          = df_extra['text'].apply(lambda x: len(str(x).split()))
    df_extra['star_sentiment']      = df_extra['rating'].apply(map_star_sentiment)
    df_extra['rough_text_sentiment']= df_extra['text'].apply(keyword_sentiment)
    df_extra['is_mismatch']         = df_extra.apply(compute_mismatch, axis=1)

    # Gabung dan hapus duplikat
    df = pd.concat([df, df_extra], ignore_index=True)
    df = df.drop_duplicates(subset=['text', 'app'], keep='first').reset_index(drop=True)

    print(f'\nTotal review setelah iterasi {iteration}: {len(df):,}')

print('\n' + '=' * 60)
if len(df) >= MIN_TOTAL:
    print(f'Validasi LULUS: Total review = {len(df):,} (>= {MIN_TOTAL:,})')
else:
    print(f'Validasi GAGAL: Total review = {len(df):,} (< {MIN_TOTAL:,})')
print('=' * 60)


Validasi LULUS: Total review = 12,792 (>= 10,000)


### 2.11 Simpan Hasil Preprocessing

In [33]:
cols_export = [
    'app', 'app_id', 'username', 'rating', 'text', 'text_original',
    'thumbs_up', 'date', 'reply', 'lang', 'word_count',
    'star_sentiment', 'rough_text_sentiment', 'is_mismatch'
]
# Hanya ekspor kolom yang ada
cols_export = [c for c in cols_export if c in df.columns]

df_clean = df[cols_export].copy()
df_clean.to_csv('clean_reviews.csv', index=False, encoding='utf-8-sig')

print(f'Data bersih berhasil disimpan ke clean_reviews.csv')
print(f'Shape final DataFrame : {df_clean.shape}')
print('\nContoh data final (3 baris):')
display(df_clean.head(3))

Data bersih berhasil disimpan ke clean_reviews.csv
Shape final DataFrame : (12792, 14)

Contoh data final (3 baris):


,app,app_id,username,rating,text,text_original,thumbs_up,date,reply,lang,word_count,star_sentiment,rough_text_sentiment,is_mismatch
0,Gojek,com.gojek.app,Pengguna Google,5,aplikasi bagus bangga buatan indonesia,aplikasi bagus bangga buatan Indonesia,0,2026-05-30 15:36:01,NaN,id,5,positive,positive,0
1,Gojek,com.gojek.app,Pengguna Google,5,saya pernah membayar 2 kali dan dikembalikan🥳 ...,saya pernah membayar 2 kali dan dikembalikan🥳 ...,0,2026-05-30 15:22:14,NaN,id,11,positive,neutral,0
2,Gojek,com.gojek.app,Pengguna Google,5,"gojek emang nyaman kemana "" aman","gojek emang nyaman kemana "" aman",0,2026-05-30 15:16:47,NaN,id,6,positive,positive,0


## Bagian 3: Eksplorasi Data

### 3.1 Distribusi Rating per Aplikasi

In [34]:
print('DISTRIBUSI RATING PER APLIKASI')
print('=' * 60)

rating_dist = df_clean.groupby(['app', 'rating']).size().unstack(fill_value=0)
rating_dist.columns = [f'Rating {c}' for c in rating_dist.columns]
rating_dist['Total'] = rating_dist.sum(axis=1)

display(rating_dist)

print('\nRata-rata rating per aplikasi:')
avg_rating = df_clean.groupby('app')['rating'].mean().round(2)
print(avg_rating.to_string())

DISTRIBUSI RATING PER APLIKASI


,Rating 1,Rating 2,Rating 3,Rating 4,Rating 5,Total
app,,,,,,
BCA Mobile,1649,360,311,173,596,3089
DANA,738,158,150,134,844,2024
Gojek,880,163,146,140,800,2129
Shopee,769,139,131,132,1342,2513
Tokopedia,1808,234,225,140,630,3037



Rata-rata rating per aplikasi:
app
BCA Mobile    2.26
DANA          3.09
Gojek         2.91
Shopee        3.45
Tokopedia     2.19


### 3.2 Distribusi is_mismatch per Aplikasi

In [35]:
print('DISTRIBUSI IS_MISMATCH PER APLIKASI')
print('=' * 60)

mismatch_tbl = df_clean.groupby('app')['is_mismatch'].agg(
    Total='count',
    Mismatch='sum'
).assign(Konsisten=lambda x: x['Total'] - x['Mismatch'])

mismatch_tbl['Pct_Mismatch (%)'] = (mismatch_tbl['Mismatch'] / mismatch_tbl['Total'] * 100).round(2)

display(mismatch_tbl)

print(f'\nTotal mismatch keseluruhan : {df_clean["is_mismatch"].sum():,}')
print(f'Persentase mismatch         : {df_clean["is_mismatch"].mean() * 100:.2f}%')

DISTRIBUSI IS_MISMATCH PER APLIKASI


,Total,Mismatch,Konsisten,Pct_Mismatch (%)
app,,,,
BCA Mobile,3089,456,2633,14.76
DANA,2024,273,1751,13.49
Gojek,2129,209,1920,9.82
Shopee,2513,271,2242,10.78
Tokopedia,3037,544,2493,17.91



Total mismatch keseluruhan : 1,753
Persentase mismatch         : 13.70%


### 3.3 Rata-rata Word Count per Segmen Panjang Teks

In [36]:
def text_segment(wc):
    if wc < 20:
        return 'Pendek (< 20 kata)'
    elif wc <= 50:
        return 'Sedang (20-50 kata)'
    else:
        return 'Panjang (> 50 kata)'

df_clean['text_segment'] = df_clean['word_count'].apply(text_segment)

print('RATA-RATA WORD COUNT PER SEGMEN TEKS')
print('=' * 60)

segment_stats = df_clean.groupby('text_segment').agg(
    Jumlah_Review=('word_count', 'count'),
    Rata_rata_Word_Count=('word_count', 'mean'),
    Min_Word_Count=('word_count', 'min'),
    Max_Word_Count=('word_count', 'max')
).round(2)

# Urutkan segmen
segment_order = ['Pendek (< 20 kata)', 'Sedang (20-50 kata)', 'Panjang (> 50 kata)']
segment_stats = segment_stats.reindex([s for s in segment_order if s in segment_stats.index])

display(segment_stats)

print('\nDistribusi segmen teks:')
seg_dist = df_clean['text_segment'].value_counts()
for seg in segment_order:
    if seg in seg_dist:
        pct = seg_dist[seg] / len(df_clean) * 100
        print(f'  {seg:<25}: {seg_dist[seg]:,} review ({pct:.1f}%)')

RATA-RATA WORD COUNT PER SEGMEN TEKS


,Jumlah_Review,Rata_rata_Word_Count,Min_Word_Count,Max_Word_Count
text_segment,,,,
Pendek (< 20 kata),8481,10.33,5,19
Sedang (20-50 kata),3558,29.85,20,50
Panjang (> 50 kata),753,66.90,51,98



Distribusi segmen teks:
  Pendek (< 20 kata)       : 8,481 review (66.3%)
  Sedang (20-50 kata)      : 3,558 review (27.8%)
  Panjang (> 50 kata)      : 753 review (5.9%)


### 3.4 Contoh Review Mismatch dan Konsisten

In [37]:
pd.set_option('display.max_colwidth', 120)

cols_show = ['app', 'rating', 'star_sentiment', 'rough_text_sentiment', 'text']
cols_show = [c for c in cols_show if c in df_clean.columns]

print('5 CONTOH REVIEW MISMATCH (star_sentiment != rough_text_sentiment)')
print('=' * 80)
df_mismatch_sample = df_clean[df_clean['is_mismatch'] == 1][cols_show].head(5)
if len(df_mismatch_sample) > 0:
    display(df_mismatch_sample)
else:
    print('Tidak ada data mismatch untuk ditampilkan.')

print()
print('5 CONTOH REVIEW KONSISTEN (star_sentiment == rough_text_sentiment)')
print('=' * 80)
df_consistent_sample = df_clean[
    (df_clean['is_mismatch'] == 0) &
    (df_clean['star_sentiment'] != 'neutral') &
    (df_clean['rough_text_sentiment'] != 'neutral')
][cols_show].head(5)

if len(df_consistent_sample) > 0:
    display(df_consistent_sample)
else:
    print('Tidak ada data konsisten (non-neutral) untuk ditampilkan.')

5 CONTOH REVIEW MISMATCH (star_sentiment != rough_text_sentiment)


,app,rating,star_sentiment,rough_text_sentiment,text
12,Gojek,1,negative,positive,stop bikin kebijakan yang bentrokin costumer sama driver
13,Gojek,5,positive,negative,sudah lama pakai aplikasi ini secara umum tak mengecewakan.
23,Gojek,4,positive,negative,"saya kasih bintang 4 karena, kemarin waktu hari kamis jam 9an awalnya saya mau mesen gocar lalu karena saya tau kala..."
39,Gojek,1,negative,positive,maaf banget driver ngeselin beda lokasi katanya dia ngomong terus tadinya aku mau ngash tips 20 ribu kalo misalkan d...
50,Gojek,1,negative,positive,"untuk gojek coba chat ny di bikin lebih fast respon lagi, soal ny ketika driver gojek ngechat atau telepon suka ada ..."



5 CONTOH REVIEW KONSISTEN (star_sentiment == rough_text_sentiment)


,app,rating,star_sentiment,rough_text_sentiment,text
0,Gojek,5,positive,positive,aplikasi bagus bangga buatan indonesia
2,Gojek,5,positive,positive,"gojek emang nyaman kemana "" aman"
7,Gojek,1,negative,negative,"susah dapat driver. padahal lagi dibutuhin banget. apa harus bayar 5x lipat dulu baru di ""prioritaskan"" ?"
8,Gojek,1,negative,negative,sumpah parah gopay udah foto ktp tapi ditolak apaa. ini susah jangan bikin gopay dah kawan hindari smua kawan kawan ...
14,Gojek,5,positive,positive,sukaakk! jujur apk ini membantu bgtt


### 3.5 Ringkasan Akhir Dataset

In [38]:
print('RINGKASAN AKHIR DATASET')
print('=' * 60)
print(f'Total review final           : {len(df_clean):,}')
print(f'Jumlah aplikasi              : {df_clean["app"].nunique()}')
print(f'Rentang rating               : {df_clean["rating"].min()} - {df_clean["rating"].max()}')
print(f'Rata-rata word count         : {df_clean["word_count"].mean():.2f} kata')
print(f'Total review mismatch        : {df_clean["is_mismatch"].sum():,}')
print(f'Persentase mismatch          : {df_clean["is_mismatch"].mean() * 100:.2f}%')
print()
print('Distribusi per aplikasi:')
print(df_clean['app'].value_counts().to_string())
print()
print('File yang dihasilkan:')
print('  - raw_reviews.csv   : data mentah dari scraping')
print('  - clean_reviews.csv : data bersih siap untuk pemodelan')

RINGKASAN AKHIR DATASET
Total review final           : 12,792
Jumlah aplikasi              : 5
Rentang rating               : 1 - 5
Rata-rata word count         : 19.09 kata
Total review mismatch        : 1,753
Persentase mismatch          : 13.70%

Distribusi per aplikasi:
app
BCA Mobile    3089
Tokopedia     3037
Shopee        2513
Gojek         2129
DANA          2024

File yang dihasilkan:
  - raw_reviews.csv   : data mentah dari scraping
  - clean_reviews.csv : data bersih siap untuk pemodelan
